In [9]:
!pip install psycopg2-binary

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 15.6 MB/s eta 0:00:0000:01m


In [22]:
# Imports & connections
import pandas as pd
import os
import matplotlib.pyplot as plt
from sqlalchemy import create_engine
from dotenv import load_dotenv

load_dotenv()
DB_URL = os.getenv('DATABASE_URL')
engine = create_engine(DB_URL)

In [62]:
# Which cause, for which the airport is made accountable, leads to how much delay by country per year?
query_delay_cause_airport = """
SELECT
    state_name,
    year,
    ROUND(SUM(dly_apt_arr_a_1), 0)  AS delay_atc_capacity,
    ROUND(SUM(dly_apt_arr_c_1), 0)  AS delay_staffing,
    ROUND(SUM(dly_apt_arr_w_1), 0)  AS delay_weather,
    ROUND(SUM(dly_apt_arr_e_1), 0)  AS delay_equipment,
    ROUND(SUM(dly_apt_arr_s_1), 0)  AS delay_special_event,
    ROUND(SUM(dly_apt_arr_t_1), 0)  AS delay_routeing,
    ROUND(SUM(dly_apt_arr_g_1), 0)  AS delay_aerodrome,
    ROUND(SUM(dly_apt_arr_m_1), 0)  AS delay_met,
    ROUND(SUM(dly_apt_arr_i_1), 0)  AS delay_industrial_action,
    ROUND(SUM(dly_apt_arr_na_1), 0) AS delay_unclassified,
    ROUND(SUM(dly_apt_arr_1), 0)    AS total_delay
FROM fact_airport_delay
GROUP BY state_name, year
ORDER BY year DESC, total_delay DESC;
"""

df_delay_cause_airport = pd.read_sql(query_delay_cause_airport, engine)

In [63]:
# Analysis & Visualisation
df_delay_cause_airport.head(20)

,state_name,year,delay_atc_capacity,delay_staffing,delay_weather,delay_equipment,delay_special_event,delay_routeing,delay_aerodrome,delay_met,delay_industrial_action,delay_unclassified,total_delay
0,Netherlands,2026,23.0,691.0,203823.0,0.0,0.0,3121.0,66740.0,0.0,0.0,0.0,274398.0
1,United Kingdom,2026,187.0,4563.0,108750.0,0.0,1184.0,169.0,55253.0,0.0,0.0,0.0,176693.0
2,Spain,2026,0.0,9360.0,94936.0,0.0,3139.0,584.0,27118.0,0.0,0.0,0.0,137789.0
3,France,2026,607.0,17542.0,62662.0,1491.0,17162.0,3721.0,8686.0,2217.0,0.0,0.0,135886.0
4,Portugal,2026,0.0,564.0,69310.0,0.0,29.0,77.0,9863.0,0.0,0.0,0.0,79843.0
5,Germany,2026,0.0,329.0,65730.0,2802.0,2728.0,626.0,3783.0,0.0,0.0,0.0,75998.0
6,Switzerland,2026,0.0,1018.0,30544.0,5756.0,6588.0,0.0,12123.0,0.0,0.0,0.0,68924.0
7,Greece,2026,0.0,44730.0,3278.0,0.0,0.0,0.0,66.0,0.0,0.0,0.0,49730.0
8,Turkiye,2026,0.0,0.0,36345.0,0.0,0.0,0.0,9048.0,0.0,0.0,0.0,45393.0
9,Denmark,2026,0.0,408.0,35548.0,0.0,9141.0,0.0,203.0,0.0,0.0,0.0,45300.0


In [65]:
# Which cause, for which the airspace of the airport is made accountable, leads to how much delay by country per year?
query_delay_cause_airport = """
SELECT
    er.country_name,
    ed.year,
    ed.month_num,
    ROUND(SUM(ed.dly_ert_a_1), 0)  AS delay_atc_capacity,
    ROUND(SUM(ed.dly_ert_c_1), 0)  AS delay_staffing,
    ROUND(SUM(ed.dly_ert_w_1), 0)  AS delay_weather,
    ROUND(SUM(ed.dly_ert_e_1), 0)  AS delay_equipment,
    ROUND(SUM(ed.dly_ert_s_1), 0)  AS delay_special_event,
    ROUND(SUM(ed.dly_ert_t_1), 0)  AS delay_routeing,
    ROUND(SUM(ed.dly_ert_g_1), 0)  AS delay_aerodrome,
    ROUND(SUM(ed.dly_ert_m_1), 0)  AS delay_met,
    ROUND(SUM(ed.dly_ert_i_1), 0)  AS delay_industrial_action,
    ROUND(SUM(ed.dly_ert_na_1), 0) AS delay_unclassified,
    ROUND(SUM(ed.dly_ert_1), 0)    AS total_delay
FROM fact_enroute_delay ed
INNER JOIN dim_entity_region er USING(entity_name)
GROUP BY er.country_name, ed.year, ed.month_num
ORDER BY ed.year DESC, total_delay DESC;
"""

df_delay_cause_airport = pd.read_sql(query_delay_cause_airport, engine)

In [67]:
df_delay_cause_airport = df_delay_cause_airport.dropna(how='any')
df_delay_cause_airport.head(20)

,country_name,year,month_num,delay_atc_capacity,delay_staffing,delay_weather,delay_equipment,delay_special_event,delay_routeing,delay_aerodrome,delay_met,delay_industrial_action,delay_unclassified,total_delay
45,Spain,2026,4,0.0,179085.0,32032.0,0.0,3677.0,0.0,342.0,0.0,0.0,0.0,221826.0
46,France,2026,4,0.0,68249.0,7114.0,0.0,51274.0,9017.0,210.0,9616.0,0.0,0.0,153955.0
47,Greece,2026,4,0.0,142.0,372.0,0.0,83727.0,0.0,0.0,0.0,0.0,0.0,132382.0
48,Spain,2026,3,0.0,98717.0,12711.0,0.0,3164.0,0.0,0.0,0.0,0.0,0.0,114624.0
49,France,2026,3,0.0,50298.0,1736.0,0.0,23954.0,5381.0,420.0,4724.0,0.0,0.0,101451.0
50,France,2026,1,0.0,33437.0,1799.0,0.0,24569.0,1034.0,1.0,117.0,0.0,0.0,97933.0
51,France,2026,2,0.0,43365.0,239.0,0.0,39125.0,4177.0,517.0,2678.0,0.0,0.0,96686.0
52,Spain,2026,2,0.0,69130.0,5270.0,0.0,763.0,0.0,0.0,364.0,0.0,0.0,85974.0
53,Spain,2026,1,0.0,55561.0,2981.0,0.0,668.0,0.0,0.0,0.0,0.0,0.0,60088.0
54,Germany,2026,1,0.0,32897.0,2095.0,0.0,6842.0,0.0,0.0,418.0,0.0,0.0,42693.0


In [33]:
# en-route delay by country
query_en_route_cause = """
SELECT
    e.entity_name,
    e.entity_type,
    r.country_name,
    e.year,
    SUM(e.flt_ert_1)                                                  AS flights,
    ROUND(SUM(e.dly_ert_1), 0)                                        AS total_delay_min,
    ROUND(SUM(e.dly_ert_1) / NULLIF(SUM(e.flt_ert_1), 0), 4)        AS delay_per_flight,
    ROUND(SUM(e.dly_ert_a_1), 0)                                      AS delay_atc_capacity,
    ROUND(SUM(e.dly_ert_c_1), 0)                                      AS delay_staffing,
    ROUND(SUM(e.dly_ert_w_1), 0)                                      AS delay_weather
FROM fact_enroute_delay e
LEFT JOIN dim_entity_region r ON e.entity_name = r.entity_name
GROUP BY e.entity_name, e.entity_type, r.country_name, e.year
ORDER BY e.year DESC, total_delay_min DESC;
"""

df_en_route_cause = pd.read_sql(query_en_route_cause, engine)

In [42]:
df_en_route_cause = df_en_route_cause.dropna(how='any')
df_en_route_cause["country_name"].value_counts()

country_name
Portugal           19
Spain              11
Greece             11
Germany            11
United Kingdom     11
France             11
Switzerland        11
Cyprus             11
Italy              11
Belgium            11
Poland             11
Serbia             11
Netherlands        11
Denmark            11
Austria            11
Hungary            11
Ireland            11
Norway             11
Tuerkiye           11
Finland            11
Sweden             11
Czechia            11
Romania            10
North Macedonia    10
Bulgaria           10
Slovenia           10
Croatia             9
Latvia              9
Slovakia            9
Albania             8
Estonia             8
Malta               8
Lithuania           8
Ukraine             6
Georgia             3
Iceland             2
Moldova             2
Azerbaijan          1
Name: count, dtype: int64